In [1]:
pip install requests pandas numpy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("All libraries loaded.")

All libraries loaded.


In [3]:
import pandas as pd

raw_df = pd.read_excel("task1_raw_weather_data.xlsx")
print(f"Loaded Task 1 data: {raw_df.shape[0]} rows × {raw_df.shape[1]} columns")
print(raw_df.head())

Loaded Task 1 data: 1830 rows × 8 columns
         time  temperature_2m_max  temperature_2m_min  temperature_2m_mean  \
0  2024-01-01                32.5                24.9                 28.2   
1  2024-01-02                32.8                25.1                 28.5   
2  2024-01-03                31.0                23.6                 27.5   
3  2024-01-04                31.5                24.9                 27.3   
4  2024-01-05                31.9                24.5                 27.8   

   precipitation_sum  windspeed_10m_max  relative_humidity_2m_mean   city  
0                0.6               14.6                         80  Accra  
1                0.7               15.6                         79  Accra  
2                2.2               18.3                         79  Accra  
3                1.9               18.3                         83  Accra  
4                0.8               18.9                         81  Accra  


In [4]:
df = raw_df.copy()

missing = df.isnull().sum()
pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({"Missing Count": missing, "Missing %": pct}))

                           Missing Count  Missing %
time                                   0        0.0
temperature_2m_max                     0        0.0
temperature_2m_min                     0        0.0
temperature_2m_mean                    0        0.0
precipitation_sum                      0        0.0
windspeed_10m_max                      0        0.0
relative_humidity_2m_mean              0        0.0
city                                   0        0.0


In [6]:
numeric_cols = [
    "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
    "precipitation_sum", "windspeed_10m_max", "relative_humidity_2m_mean"
]

for col in numeric_cols:
    before = df[col].isnull().sum()
    df[col] = df.groupby("city")[col].transform(lambda x: x.fillna(x.mean()))
    after = df[col].isnull().sum()
    if before > 0:
        print(f"  {col}: filled {before} missing → using city mean")

print(f"\n Remaining NaNs: {df.isnull().sum().sum()}")


 Remaining NaNs: 0


In [7]:
before = len(df)
df = df.drop_duplicates(subset=["time", "city"])
print(f"Rows before : {before}")
print(f"Duplicates removed : {before - len(df)}")
print(f"Rows after  : {len(df)}")

Rows before : 1830
Duplicates removed : 0
Rows after  : 1830


In [8]:
# Convert date column
df["time"] = pd.to_datetime(df["time"])

# Extract time features
df["month"]      = df["time"].dt.month
df["month_name"] = df["time"].dt.strftime("%b")
df["quarter"]    = df["time"].dt.quarter
df["day_of_year"]= df["time"].dt.dayofyear

# Add temperature range feature
df["temp_range"] = df["temperature_2m_max"] - df["temperature_2m_min"]

# Round numeric columns
for col in numeric_cols:
    df[col] = df[col].round(2)

print("Formatting complete.")
print(df.dtypes)

Formatting complete.
time                         datetime64[ns]
temperature_2m_max                  float64
temperature_2m_min                  float64
temperature_2m_mean                 float64
precipitation_sum                   float64
windspeed_10m_max                   float64
relative_humidity_2m_mean             int64
city                                 object
month                                 int32
month_name                           object
quarter                               int32
day_of_year                           int32
temp_range                          float64
dtype: object


In [9]:
print(f"Shape         : {df.shape}")
print(f"Date range    : {df['time'].min().date()} → {df['time'].max().date()}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicates    : {df.duplicated().sum()}")
df.head()

Shape         : (1830, 13)
Date range    : 2024-01-01 → 2024-12-31
Missing values: 0
Duplicates    : 0


,time,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,windspeed_10m_max,relative_humidity_2m_mean,city,month,month_name,quarter,day_of_year,temp_range
0,2024-01-01,32.5,24.9,28.2,0.6,14.6,80,Accra,1,Jan,1,1,7.6
1,2024-01-02,32.8,25.1,28.5,0.7,15.6,79,Accra,1,Jan,1,2,7.7
2,2024-01-03,31.0,23.6,27.5,2.2,18.3,79,Accra,1,Jan,1,3,7.4
3,2024-01-04,31.5,24.9,27.3,1.9,18.3,83,Accra,1,Jan,1,4,6.6
4,2024-01-05,31.9,24.5,27.8,0.8,18.9,81,Accra,1,Jan,1,5,7.4


In [10]:
df.to_excel("task2_cleaned_weather_data.xlsx", index=False)
print("Saved: task2_cleaned_weather_data.xlsx")

Saved: task2_cleaned_weather_data.xlsx
